<h1>Chapter 11 - RAG Chatbot with Streamlit</h1>
<i>Building a SQL chat application using Vanna for Text-to-SQL.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch11_rag_chatbot_streamlit/11.4_sql_database_connection/vannachat.ipynb)

---

This notebook is for Chapter 11 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


### How to initialize and train Vanna for Text-to-SQL

In [ ]:
!pip install vanna
!pip install chromadb==0.4.22
!pip install "nbformat>=4.2.0"

  Using cached chromadb-0.4.22-py3-none-any.whl.metadata (7.3 kB)
  Using cached chroma-hnswlib-0.7.3.tar.gz (31 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached fastapi-0.115.14-py3-none-any.whl.metadata (27 kB)
  Using cached pulsar_client-3.7.0-cp313-cp313-win_amd64.whl.metadata (1.2 kB)
  Using cached opentelemetry_instrumentation_fastapi-0.55b1-py3-none-any.whl.metadata (2.2 kB)
  Using cached starlette-0.46.2-py3-none-any.whl.metadata (6.2 kB)
  Using cached opentelemetry_instrumentation_asgi-0.55b1-py3-none-any.whl.metadata (2.0 kB)
  Using cached opentelemetry_instrumentation-0.55b1-py3-none-any.whl.metadata (6.7 kB)
  Using cached opentelemetry_util_http-0.55b1-py3-none-any.whl.meta

  error: subprocess-exited-with-error
  
  × Building wheel for chroma-hnswlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [5 lines of output]
      running bdist_wheel
      running build
      running build_ext
      building 'hnswlib' extension
      error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/visual-cpp-build-tools/
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for chroma-hnswlib
ERROR: Failed to build installable wheels for some pyproject.toml based projects (chroma-hnswlib)


To have a database that we can query in the following steps, we first create a simple SQLite database with some sample data. SQLite is a lightweight, in-file database that is well-suited for small projects and demonstrations.

I have created a Python script for you that generates the SQLite database file and fills it with some sample data. The database models a bookstore system. It includes tables for genres, clients, staff, deliveries, authors, books, sales, and sale items. You can find the script `create_sample_database.py` in the test[book repo].

Now we can build the actual chat app, where we use Vanna to search and analyze the database using natural language.

First, we need a chat template for user input.

Let's first define the Vanna class

In [ ]:
import os
from vanna.legacy.openai.openai_chat import OpenAI_Chat
from vanna.legacy.chromadb.chromadb_vector import ChromaDB_VectorStore

class MyVanna(ChromaDB_VectorStore, OpenAI_Chat):
    def __init__(self, config=None):
        ChromaDB_VectorStore.__init__(self, config=config)
        OpenAI_Chat.__init__(self, config=config)

vn = MyVanna(config={
    "api_key": os.environ.get("OPENAI_API_KEY"),
    "model": "gpt-4o"
})

Now we connect Vanna to the database that we want to query later using the user's question.

In [ ]:
vn.connect_to_sqlite('bookstore.db')

: 

Once you have defined the class, you need to "train" Vanna. With `vn.train` you load helpful information into your vector store, which can later help the LLM to build suitable SQL queries.

Vanna recommends 3 types of training data:

1. DDL Statements: Give Vanna your database schema (table names, column names, data types, relationships).
   Example: vn.train(ddl="CREATE TABLE customers (id INT PRIMARY KEY, name TEXT)")
2. Documentation Strings: Provide general information about your business logic, terms, or data.
   Example: vn.train(documentation="Revenue is calculated as sales minus returns.")
3. SQL Statements: Feed Vanna existing, useful SQL queries from your organization.
   Example: vn.train(sql="SELECT id, name FROM employees WHERE department = 'Sales'")
4. Question-SQL Pairs: This is the most effective. Pair natural language questions with their correct SQL answers.
   Example: vn.train(question="Show me all active users", sql="SELECT * FROM users WHERE status = 'active'")

Now we want to adapt Vanna to our database, essentially training it to understand the content of the database. You only need to train once. Do not train again unless you want to add more training data.

The code below retrieves the database schema from our SQLite system table and trains Vanna with the schema information to understand the database structure.

In [4]:
df_ddl = vn.run_sql("SELECT type, sql FROM sqlite_master WHERE sql is not null")

for ddl in df_ddl['sql'].to_list():
  vn.train(ddl=ddl)

Adding ddl: CREATE TABLE genres (
    GenreID INTEGER PRIMARY KEY AUTOINCREMENT,
    GenreName TEXT,
    Description TEXT
)
Adding ddl: CREATE TABLE sqlite_sequence(name,seq)
Adding ddl: CREATE TABLE clients (
    ClientID INTEGER PRIMARY KEY AUTOINCREMENT,
    ClientName TEXT,
    ContactEmail TEXT,
    Street TEXT,
    Town TEXT,
    ZipCode TEXT,
    Country TEXT
)
Adding ddl: CREATE TABLE staff (
    StaffID INTEGER PRIMARY KEY AUTOINCREMENT,
    Surname TEXT,
    GivenName TEXT,
    HireDate TEXT,
    ProfilePic TEXT,
    Bio TEXT
)
Adding ddl: CREATE TABLE deliveries (
    DeliveryID INTEGER PRIMARY KEY AUTOINCREMENT,
    DeliveryCompany TEXT,
    ContactNumber TEXT
)
Adding ddl: CREATE TABLE authors (
    AuthorID INTEGER PRIMARY KEY AUTOINCREMENT,
    AuthorName TEXT,
    Bio TEXT
)
Adding ddl: CREATE TABLE books (
    BookID INTEGER PRIMARY KEY AUTOINCREMENT,
    Title TEXT,
    AuthorID INTEGER,
    GenreID INTEGER,
    Format TEXT,
    Price REAL,
    FOREIGN KEY (GenreID) R

Once everything is trained, we use our Vanna class to translate the user's question.

In [5]:
user_input = "What are the top 5 best-selling books?"

sql_query = vn.generate_sql(user_input)
result = vn.run_sql(sql_query)

[{'role': 'system', 'content': "You are a SQLite expert. \n===Tables \nCREATE TABLE books (\n    BookID INTEGER PRIMARY KEY AUTOINCREMENT,\n    Title TEXT,\n    AuthorID INTEGER,\n    GenreID INTEGER,\n    Format TEXT,\n    Price REAL,\n    FOREIGN KEY (GenreID) REFERENCES genres(GenreID),\n    FOREIGN KEY (AuthorID) REFERENCES authors(AuthorID)\n)\n\nCREATE TABLE sale_items (\n    SaleItemID INTEGER PRIMARY KEY AUTOINCREMENT,\n    SaleID INTEGER,\n    BookID INTEGER,\n    Quantity INTEGER,\n    FOREIGN KEY (SaleID) REFERENCES sales(SaleID),\n    FOREIGN KEY (BookID) REFERENCES books(BookID)\n)\n\nCREATE TABLE sales (\n    SaleID INTEGER PRIMARY KEY AUTOINCREMENT,\n    ClientID INTEGER,\n    StaffID INTEGER,\n    SaleDate TEXT,\n    DeliveryID INTEGER,\n    FOREIGN KEY (ClientID) REFERENCES clients(ClientID),\n    FOREIGN KEY (StaffID) REFERENCES staff(StaffID),\n    FOREIGN KEY (DeliveryID) REFERENCES deliveries(DeliveryID)\n)\n\nCREATE TABLE genres (\n    GenreID INTEGER PRIMARY KEY 

In [6]:
print(sql_query)
result

SELECT b.Title, SUM(si.Quantity) AS TotalSold
FROM books b
JOIN sale_items si ON b.BookID = si.BookID
GROUP BY b.BookID
ORDER BY TotalSold DESC
LIMIT 5;


,Title,TotalSold
0,Mystic River,2
1,The Lost Galaxy,1


Um das ganze nicht immer wieder trainieren zu müssen, speicher ich die t